(parameters)=

# Working with the `Parameters` Class

`Gal3D` uses the {py:class}`Parameters <gal3d.optimization.parameter.Parameters>` class to manage and store model parameters.

## Basic Usage

We can define a set of parameters as follows:

In [1]:
from gal3d.optimization.parameter import Parameters

param = Parameters(a=3,b=2,c=1)
param

Parameters(
  a  = 3.000  [-inf, inf]
  b  = 2.000  [-inf, inf]
  c  = 1.000  [-inf, inf]
)

Each parameter value is represented by a {py:class}`Parameter <gal3d.optimization.parameter.Parameter>` object.

The lower bound (`lb`) and upper bound (`ub`) of each parameter can be specified explicitly. If they are not given, they default to negative infinity and positive infinity, respectively.

In [2]:
print(param['a'].__class__.__name__)
param['a']

Parameter


Parameter(3.000 ∈ [-inf, inf])

You can modify `lb` and `ub` directly:

In [3]:
param['a'].lb = 0

You can also set bounds in batch using the `set_ub` and `set_lb` methods:

In [ ]:
param.set_ub(b=0,c=0)
print(param['b'],param['c'])

Parameter(2.000 ∈ [-inf, 0.000]) Parameter(1.000 ∈ [-inf, 0.000])


## Adding Extra Information and Derived Parameters

You can add derived parameters using the `add_derived` method. The names of all derived parameters can be obtained with `derived_keys`:

In [ ]:
param.add_derived("eps_ab",lambda p: 1 - p['b']/p['a'])
print(param.derived_keys(),"eps_ab: ", param['eps_ab'])

dict_keys(['eps_ab']) eps_ab:  0.33333333333333337


You can add additional information using the `add_info` method. The names of all such entries can be obtained with `info_keys`:

In [6]:
param.add_info(mass = 10)
print(param.info_keys(),"mass: ", param['mass'])

dict_keys(['mass']) mass:  10


The list of basic parameter names can be obtained using `parameter_keys`. These are all {py:class}`Parameter <gal3d.optimization.parameter.Parameter>` objects:

In [7]:
param.parameter_keys()

dict_keys(['a', 'b', 'c'])

## Adding Parameter Constraints

You can add equality constraints to basic parameters using the `add_equal_constraints` method:

In [ ]:
param = Parameters(a=3,b=2,c=1)
param.set_lb(a=0.3,b=0.2,c=0.1)

param.add_equal_constraints(c = lambda p: p['b'])
param['c']

2.0

In [9]:
param

Parameters(
  a  = 3.000  [0.300, inf]
  b  = 2.000  [0.200, inf]
)

Parameters with equality constraints are removed from `parameter_keys`:

In [10]:
param.parameter_keys()

dict_keys(['a', 'b'])

You can remove previously added equality constraints using the `del_equal_constraints` method. The corresponding parameters will then reappear in `parameter_keys` and return to their original unconstrained state:

In [11]:
param.del_equal_constraints("c")

param.parameter_keys()
print(param['c'])

Parameter(1.000 ∈ [0.100, inf])


## Parameter Priority

In the `Parameters` class, values are divided into basic parameters, constrained parameters, derived parameters, and additional information.

Their priority, from highest to lowest, is:

constrained parameters > basic parameters > derived parameters > additional information

That is, when retrieving a value, `Parameters` searches in `constraint_keys`, `parameter_keys`, `derived_keys`, and `info_keys` in that order.

An example is shown below:

Create a parameter set and add `eps_ab` as additional information:

In [12]:
params = Parameters(a=2, b=1)
params.add_info(eps_ab=5)
params['eps_ab']

5

Add `eps_ab` as a derived parameter. In this case, the retrieved value is the derived one:

In [ ]:
params.add_derived("eps_ab", lambda p: 1 - p['b']/p['a'])
params['eps_ab']

0.5

Add `eps_ab` directly as a basic parameter. In this case, the retrieved value is the basic parameter value:

In [ ]:
params['eps_ab'] = 0.4
params['eps_ab']

Parameter(0.400 ∈ [-inf, inf])

Add `eps_ab` as a constrained parameter. In this case, the retrieved value is the constrained one:

In [ ]:
params.add_equal_constraints(eps_ab=lambda k: 0.25)
params['eps_ab']

0.25

After removing the constraint, the retrieved value becomes the basic parameter value again:

In [ ]:
params.del_equal_constraints("eps_ab")
params['eps_ab']

Parameter(0.400 ∈ [-inf, inf])